In [1]:
#importing libraries
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

INPUT_FILE = "semantic_relevance_results.json"

# LOADING RESULTS
with open(INPUT_FILE, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("\n✅ Loaded results")
print(df.head())


# SUMMARY STATISTICS

print("\n📊 SUMMARY STATISTICS\n")

summary = {
    "Metric": ["Mean", "Median", "Std", "Min", "Max"],

    "Agg Title": [
        df["agg_title_similarity"].mean(),
        df["agg_title_similarity"].median(),
        df["agg_title_similarity"].std(),
        df["agg_title_similarity"].min(),
        df["agg_title_similarity"].max(),
    ],

    "Agg Description": [
        df["agg_description_similarity"].mean(),
        df["agg_description_similarity"].median(),
        df["agg_description_similarity"].std(),
        df["agg_description_similarity"].min(),
        df["agg_description_similarity"].max(),
    ],

    "Avg Title": [
        df["avg_title_similarity"].mean(),
        df["avg_title_similarity"].median(),
        df["avg_title_similarity"].std(),
        df["avg_title_similarity"].min(),
        df["avg_title_similarity"].max(),
    ],

    "Avg Description": [
        df["avg_description_similarity"].mean(),
        df["avg_description_similarity"].median(),
        df["avg_description_similarity"].std(),
        df["avg_description_similarity"].min(),
        df["avg_description_similarity"].max(),
    ]
}

summary_df = pd.DataFrame(summary)
print(summary_df)

# HISTOGRAMS

# Mean-pooled aggregated: Title vs Description
plt.figure()

plt.hist(
    df["agg_title_similarity"],
    bins=50,
    alpha=0.5,
    label="Title"
)

plt.hist(
    df["agg_description_similarity"],
    bins=50,
    alpha=0.5,
    label="Description"
)

plt.legend()
plt.xlabel("Aggregated Semantic Similarity")
plt.ylabel("Frequency")
plt.title("Mean-Pooled Aggregated Relevance: Title vs Description")

plt.savefig("hist_agg_title_vs_description.png")
plt.close()

# Average: Title vs Description
plt.figure()

plt.hist(
    df["avg_title_similarity"],
    bins=50,
    alpha=0.5,
    label="Title"
)

plt.hist(
    df["avg_description_similarity"],
    bins=50,
    alpha=0.5,
    label="Description"
)

plt.legend()
plt.xlabel("Average Keyword-Level Similarity")
plt.ylabel("Frequency")
plt.title("Average Keyword Relevance: Title vs Description")

plt.savefig("hist_avg_title_vs_description.png")
plt.close()

# BOXPLOTS

plt.figure(figsize=(10, 6))

box_data = [
    df["agg_title_similarity"],
    df["agg_description_similarity"],
    df["avg_title_similarity"],
    df["avg_description_similarity"]
]

labels = [
    "Agg Title",
    "Agg Description",
    "Avg Title",
    "Avg Description"
]

bp = plt.boxplot(
    box_data,
    tick_labels=labels,
    patch_artist=True
)

medians = [
    df["agg_title_similarity"].median(),
    df["agg_description_similarity"].median(),
    df["avg_title_similarity"].median(),
    df["avg_description_similarity"].median()
]

for i, median in enumerate(medians, start=1):

    plt.text(
        i + 0.08,
        median,
        f"{median:.2f}",
        verticalalignment='center',
        fontsize=9
    )

plt.ylabel("Semantic Similarity")
plt.title("Semantic Relevance Comparison")

plt.savefig(
    "boxplot_semantic_relevance.png",
    bbox_inches='tight'
)

plt.close()

# THRESHOLD ANALYSIS

def categorize(score):
    if score >= 0.75:
        return "High"
    elif score >= 0.5:
        return "Moderate"
    else:
        return "Low"

# Aggregated Title
df["agg_title_category"] = df["agg_title_similarity"].apply(categorize)

# Aggregated Description
df["agg_description_category"] = df["agg_description_similarity"].apply(categorize)

title_category_counts = (
    df["agg_title_category"]
    .value_counts(normalize=True) * 100
)

description_category_counts = (
    df["agg_description_category"]
    .value_counts(normalize=True) * 100
)

print("\n📊 AGGREGATED TITLE DISTRIBUTION (%)\n")
print(title_category_counts)

print("\n📊 AGGREGATED DESCRIPTION DISTRIBUTION (%)\n")
print(description_category_counts)

# SIMILARITY RANGE ANALYSIS

print("\n📊 SIMILARITY RANGE DISTRIBUTION\n")

bins = [i/10 for i in range(11)]
labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

# Title bins
df["title_similarity_bin"] = pd.cut(
    df["agg_title_similarity"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Description bins
df["description_similarity_bin"] = pd.cut(
    df["agg_description_similarity"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

title_bin_counts = (
    df["title_similarity_bin"]
    .value_counts()
    .sort_index()
)

description_bin_counts = (
    df["description_similarity_bin"]
    .value_counts()
    .sort_index()
)

title_bin_percent = (
    title_bin_counts / len(df)
) * 100

description_bin_percent = (
    description_bin_counts / len(df)
) * 100

range_df = pd.DataFrame({
    "Range": title_bin_counts.index,
    "Title Percentage": title_bin_percent.values,
    "Description Percentage": description_bin_percent.values
})

print(range_df)

# RANGE BAR PLOT

x = np.arange(len(labels))
width = 0.35

plt.figure()

plt.bar(
    x - width/2,
    title_bin_percent.values,
    width,
    label="Title"
)

plt.bar(
    x + width/2,
    description_bin_percent.values,
    width,
    label="Description"
)

plt.xticks(x, labels, rotation=45)

plt.xlabel("Similarity Range")
plt.ylabel("Percentage of Datasets")

plt.title("Mean-Pooled Aggregated Semantic Relevance Distribution")

plt.legend()

plt.tight_layout()

plt.savefig("range_distribution_title_vs_description.png")
plt.close()

# SCATTER: TITLE VS DESCRIPTION

plt.figure()

plt.scatter(
    df["agg_title_similarity"],
    df["agg_description_similarity"],
    alpha=0.5
)

plt.xlabel("Aggregated Title Similarity")
plt.ylabel("Aggregated Description Similarity")

plt.title("Title vs Description Semantic Relevance")

plt.savefig("scatter_title_vs_description.png")
plt.close()

# SCATTER: KEYWORD COUNT VS RELEVANCE

plt.figure()

plt.scatter(
    df["num_keywords"],
    df["agg_title_similarity"],
    alpha=0.5,
    label="Title"
)

plt.scatter(
    df["num_keywords"],
    df["agg_description_similarity"],
    alpha=0.5,
    label="Description"
)

plt.xlabel("Number of Keywords")
plt.ylabel("Aggregated Semantic Similarity")

plt.title("Keyword Count vs Semantic Relevance")

plt.legend()

plt.savefig("scatter_keywords_vs_relevance.png")
plt.close()

# GAP ANALYSIS

df["title_gap"] = (
    df["agg_title_similarity"]
    - df["avg_title_similarity"]
)

df["description_gap"] = (
    df["agg_description_similarity"]
    - df["avg_description_similarity"]
)

print("\n📉 TITLE GAP ANALYSIS\n")
print(df["title_gap"].describe())

print("\n📉 DESCRIPTION GAP ANALYSIS\n")
print(df["description_gap"].describe())

# TOP & BOTTOM DATASETS

top_title = df.sort_values(
    "agg_title_similarity",
    ascending=False
).head(10)

bottom_title = df.sort_values(
    "agg_title_similarity",
    ascending=True
).head(10)

top_description = df.sort_values(
    "agg_description_similarity",
    ascending=False
).head(10)

bottom_description = df.sort_values(
    "agg_description_similarity",
    ascending=True
).head(10)

print("\n🏆 TOP TITLE DATASETS\n")
print(top_title)

print("\n⚠️ BOTTOM TITLE DATASETS\n")
print(bottom_title)

print("\n🏆 TOP DESCRIPTION DATASETS\n")
print(top_description)

print("\n⚠️ BOTTOM DESCRIPTION DATASETS\n")
print(bottom_description)

# CORRELATION

print("\n🔗 CORRELATION MATRIX\n")

correlation = df[[
    "agg_title_similarity",
    "agg_description_similarity",
    "avg_title_similarity",
    "avg_description_similarity",
    "num_keywords"
]].corr()

print(correlation)

# SAVE OUTPUTS

summary_df.to_csv("semantic_rel_summary_statistics.csv", index=False)

title_category_counts.to_csv(
    "title_category_distribution.csv"
)

description_category_counts.to_csv(
    "description_category_distribution.csv"
)

range_df.to_csv(
    "similarity_range_distribution.csv",
    index=False
)

top_title.to_csv(
    "top_title_datasets.csv",
    index=False
)

bottom_title.to_csv(
    "bottom_title_datasets.csv",
    index=False
)

top_description.to_csv(
    "top_description_datasets.csv",
    index=False
)

bottom_description.to_csv(
    "bottom_description_datasets.csv",
    index=False
)

print("\n💾 Files saved:")
print("- semantic_rel_summary_statistics.csv")
print("- title_category_distribution.csv")
print("- description_category_distribution.csv")
print("- similarity_range_distribution.csv")
print("- top_title_datasets.csv")
print("- bottom_title_datasets.csv")
print("- top_description_datasets.csv")
print("- bottom_description_datasets.csv")
print("- hist_agg_title_vs_description.png")
print("- hist_avg_title_vs_description.png")
print("- boxplot_semantic_relevance.png")
print("- range_distribution_title_vs_description.png")
print("- scatter_title_vs_description.png")
print("- scatter_keywords_vs_relevance.png")

print("\n✅ Analysis complete!")


✅ Loaded results
                                          dataset_id  agg_title_similarity  \
0  http://data.europa.eu/88u/dataset/taxi-and-pri...              0.441874   
1  http://data.europa.eu/88u/dataset/qics-data-2d...              0.368456   
2  http://data.europa.eu/88u/dataset/free-school-...              0.428973   
3  http://data.europa.eu/88u/dataset/farm-census-...              0.472729   
4  http://data.europa.eu/88u/dataset/register-of-...              0.754179   

   agg_description_similarity  avg_title_similarity  \
0                    0.408187              0.270456   
1                    0.401442              0.196217   
2                    0.309691              0.258437   
3                    0.524632              0.271891   
4                    0.625851              0.488089   

   avg_description_similarity  num_keywords  
0                    0.249837            12  
1                    0.213782             6  
2                    0.186574             9 